In [30]:
import pandas as pd
import re

In [31]:
import pandas as pd
import re
from itertools import product

def parse_prerequisites(prereq_text):
    """
    Parsea el texto de prerrequisitos y devuelve una lista de opciones.
    Cada opción es una lista de materias que deben cumplirse (AND).
    """
    if pd.isna(prereq_text) or prereq_text.strip() == '':
        return []
    
    # Limpiar el texto
    text = str(prereq_text).strip()
    
    # Reemplazar O y A por símbolos más fáciles de procesar
    text = text.replace(' O ', ' | ')
    text = text.replace(' A ', ' & ')
    
    def evaluate_expression(expr):
        """
        Evalúa recursivamente expresiones con paréntesis
        """
        expr = expr.strip()
        temp_storage = {}
        temp_counter = 0
        
        # Procesar paréntesis de adentro hacia afuera
        while '(' in expr:
            # Encontrar el paréntesis más interno
            start = -1
            for i, char in enumerate(expr):
                if char == '(':
                    start = i
                elif char == ')' and start != -1:
                    # Procesar el contenido entre paréntesis
                    inner_content = expr[start+1:i]
                    inner_options = process_or_expression(inner_content)
                    
                    # Crear un placeholder temporal
                    temp_id = f"TEMP_{temp_counter}"
                    temp_storage[temp_id] = inner_options
                    temp_counter += 1
                    
                    # Reemplazar el paréntesis con el placeholder
                    expr = expr[:start] + temp_id + expr[i+1:]
                    break
        
        # Procesar la expresión final
        final_options = process_or_expression(expr)
        
        # Expandir todos los placeholders
        expanded_options = expand_all_temps(final_options, temp_storage)
        
        return expanded_options
    
    def process_or_expression(expr):
        """
        Procesa expresiones dividiendo por OR (|)
        """
        # Dividir por OR
        or_parts = [part.strip() for part in expr.split('|')]
        result = []
        
        for part in or_parts:
            # Procesar AND
            and_parts = [p.strip() for p in part.split('&') if p.strip()]
            if and_parts:
                result.append(and_parts)
        
        return result
    
    def expand_all_temps(options, temp_storage):
        """
        Expande todos los placeholders temporales en todas las opciones
        """
        expanded_options = []
        
        for option in options:
            # Verificar si esta opción contiene placeholders
            temp_indices = []
            regular_parts = []
            
            for i, part in enumerate(option):
                if part.startswith('TEMP_') and part in temp_storage:
                    temp_indices.append((i, part))
                else:
                    regular_parts.append((i, part))
            
            if not temp_indices:
                # No hay placeholders, agregar la opción tal como está
                expanded_options.append(option)
            else:
                # Hay placeholders, necesitamos expandir todas las combinaciones
                expanded_combinations = expand_temp_combinations(option, temp_indices, temp_storage)
                expanded_options.extend(expanded_combinations)
        
        return expanded_options
    
    def expand_temp_combinations(base_option, temp_indices, temp_storage):
        """
        Expande todas las combinaciones posibles de los placeholders temporales
        """
        if not temp_indices:
            return [base_option]
        
        # Obtener todas las opciones para cada placeholder
        temp_options_list = []
        for idx, temp_id in temp_indices:
            temp_options_list.append(temp_storage[temp_id])
        
        # Generar todas las combinaciones posibles
        all_combinations = list(product(*temp_options_list))
        
        results = []
        for combination in all_combinations:
            # Crear una nueva opción reemplazando los placeholders
            new_option = base_option.copy()
            
            for i, (temp_idx, temp_id) in enumerate(temp_indices):
                # Reemplazar el placeholder con la opción seleccionada de la combinación
                selected_temp_option = combination[i]
                # Reemplazar el elemento en temp_idx con todos los elementos de selected_temp_option
                new_option[temp_idx:temp_idx+1] = selected_temp_option
            
            results.append(new_option)
        
        return results
    
    try:
        options = evaluate_expression(text)
        return options
    except Exception as e:
        print(f"Error procesando '{text}': {e}")
        return []

def process_prerequisites_file(file_path):
    """
    Procesa el archivo de prerrequisitos y genera las nuevas columnas
    """
    # Leer el archivo
    try:
        if file_path.endswith('.csv'):
            df = pd.read_csv(file_path)
        else:
            df = pd.read_excel(file_path)
    except Exception as e:
        raise ValueError(f"Error leyendo el archivo: {e}")
    
    # Verificar que las columnas necesarias existen
    if 'prereq completo' not in df.columns:
        available_cols = list(df.columns)
        raise ValueError(f"La columna 'prereq completo' no se encuentra en el archivo. Columnas disponibles: {available_cols}")
    
    print("Procesando prerrequisitos...")
    
    # Procesar cada fila
    all_options = []
    max_options = 0
    
    for idx, row in df.iterrows():
        prereq_text = row['prereq completo']
        options = parse_prerequisites(prereq_text)
        
        # Convertir las opciones a strings con &
        formatted_options = []
        for option in options:
            if isinstance(option, list) and len(option) > 0:
                # Filtrar elementos vacíos y limpiar
                clean_option = []
                for item in option:
                    if isinstance(item, str) and item.strip():
                        clean_option.append(item.strip())
                
                if len(clean_option) > 1:
                    formatted_options.append(' & '.join(clean_option))
                elif len(clean_option) == 1:
                    formatted_options.append(clean_option[0])
        
        all_options.append(formatted_options)
        max_options = max(max_options, len(formatted_options))
        
        # Mostrar progreso cada 25 filas
        if (idx + 1) % 25 == 0:
            print(f"Procesadas {idx + 1} filas...")
    
    print(f"Creando {max_options} columnas de opciones...")
    
    # Crear las nuevas columnas
    for i in range(max_options):
        col_name = f'Opcion_Prereq_{i+1}'
        df[col_name] = ''
        
        for idx, options in enumerate(all_options):
            if i < len(options):
                df.loc[idx, col_name] = options[i]
    
    return df

def test_complex_case():
    """
    Prueba específica para el caso problemático
    """
    test_case = "( IIN4315 | IIN4319 ) & IST7122 & ( IGL7080 | IGL4925 | FRA7021 | IGL8530 | ALE7091 ) | IGL8525"
    
    print("Prueba del caso complejo:")
    print("=" * 80)
    print(f"Input: {test_case}")
    
    options = parse_prerequisites(test_case)
    
    print(f"Número de opciones generadas: {len(options)}")
    
    for i, option in enumerate(options, 1):
        if isinstance(option, list) and len(option) > 0:
            clean_option = [item.strip() for item in option if isinstance(item, str) and item.strip()]
            if len(clean_option) > 1:
                formatted = ' & '.join(clean_option)
            elif len(clean_option) == 1:
                formatted = clean_option[0]
            else:
                formatted = "Opción vacía"
        else:
            formatted = "Opción inválida"
        
        print(f"Opción {i}: {formatted}")
    
    print("=" * 80)

def show_examples():
    """
    Muestra ejemplos de cómo funciona el parser
    """
    examples = [
        "MAT1101 O INTE 00",
        "MAT1111 A ( FIS1023 O CSV0040 ) O INTE 00",
        "FIS1001 A MAT1001",
        "( QUI1001 O BIO1001 ) A MAT1002",
        "( IIN4315 | IIN4319 ) & IST7122 & ( IGL7080 | IGL4925 ) | IGL8525"
    ]
    
    print("Ejemplos de funcionamiento:")
    print("=" * 60)
    
    for example in examples:
        print(f"Input: '{example}'")
        options = parse_prerequisites(example)
        
        formatted_options = []
        for option in options:
            if isinstance(option, list) and len(option) > 0:
                clean_option = [item.strip() for item in option if isinstance(item, str) and item.strip()]
                if len(clean_option) > 1:
                    formatted_options.append(' & '.join(clean_option))
                elif len(clean_option) == 1:
                    formatted_options.append(clean_option[0])
        
        print(f"Opciones generadas: {len(formatted_options)}")
        for i, opt in enumerate(formatted_options, 1):
            print(f"  Opción {i}: {opt}")
        print("-" * 60)



In [32]:
def main():
    """
    Función principal para ejecutar el procesamiento
    """
    print("Procesador de Prerrequisitos Universitarios")
    print("=" * 50)
    
    # Opción para mostrar ejemplos o prueba compleja
    choice = input("¿Qué deseas hacer?\n1. Ver ejemplos\n2. Probar caso complejo\n3. Procesar archivo\nElige (1/2/3): ").strip()
    
    if choice == '1':
        show_examples()
        print()
    elif choice == '2':
        test_complex_case()
        print()
        return
    
    # Solicitar la ruta del archivo
    file_path = input("Ingresa la ruta del archivo (CSV o Excel): ").strip()
    
    if not file_path:
        print("No se proporcionó una ruta de archivo.")
        return None
    
    try:
        # Procesar el archivo
        print(f"\nProcesando archivo: {file_path}")
        df_result = process_prerequisites_file(file_path)
        
        # Mostrar información sobre el resultado
        print(f"\n✅ Procesamiento completado!")
        print(f"Total de filas: {len(df_result)}")
        
        # Contar cuántas columnas de opciones se crearon
        option_cols = [col for col in df_result.columns if col.startswith('Opcion_Prereq_')]
        print(f"Columnas de opciones creadas: {len(option_cols)}")
        
        # Mostrar filas con prerrequisitos (no vacías)
        non_empty_prereqs = df_result[df_result['prereq completo'].notna() & (df_result['prereq completo'] != '')]
        if len(non_empty_prereqs) > 0:
            print(f"\nFilas con prerrequisitos: {len(non_empty_prereqs)}")
            print("\nPrimeras 3 filas con prerrequisitos:")
            cols_to_show = []
            if 'Smbarul_Key_Rule' in df_result.columns:
                cols_to_show.append('Smbarul_Key_Rule')
            cols_to_show.extend(['prereq completo'] + option_cols)
            
            available_cols = [col for col in cols_to_show if col in df_result.columns]
            print(non_empty_prereqs[available_cols].head(3).to_string())
        
        # Guardar el resultado
        file_parts = file_path.rsplit('.', 1)
        if len(file_parts) == 2:
            output_path = file_parts[0] + '_procesado.' + file_parts[1]
        else:
            output_path = file_path + '_procesado.csv'
        
        if file_path.endswith('.csv') or not file_path.endswith(('.xlsx', '.xls')):
            df_result.to_csv(output_path, index=False)
        else:
            df_result.to_excel(output_path, index=False)
        
        print(f"\n💾 Archivo guardado como: {output_path}")
        
        return df_result
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return None


In [33]:


if __name__ == "__main__":
    main()

Procesador de Prerrequisitos Universitarios

Procesando archivo: para trabajar pre req asig sistemas.xlsx
Procesando prerrequisitos...
Procesadas 25 filas...
Procesadas 50 filas...
Creando 11 columnas de opciones...

✅ Procesamiento completado!
Total de filas: 52
Columnas de opciones creadas: 11

Filas con prerrequisitos: 26

Primeras 3 filas con prerrequisitos:
  Smbarul_Key_Rule    prereq completo Opcion_Prereq_1 Opcion_Prereq_2 Opcion_Prereq_3 Opcion_Prereq_4 Opcion_Prereq_5 Opcion_Prereq_6 Opcion_Prereq_7 Opcion_Prereq_8 Opcion_Prereq_9 Opcion_Prereq_10 Opcion_Prereq_11
5          FIS1023  MAT1101 O INTE 00         MAT1101         INTE 00                                                                                                                                                  
6          CAS3030  CAS3020 O LEY4715         CAS3020         LEY4715                                                                                                                                      